In [ ]:
!pip install -q kaggle

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json

!kaggle datasets download -d emmarex/plantdisease -p /content

cp: cannot stat 'kaggle.json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory
Dataset URL: https://www.kaggle.com/datasets/emmarex/plantdisease
License(s): unknown
100% 658M/658M [00:05<00:00, 118MB/s]



In [ ]:
import random
import shutil
import zipfile
from pathlib import Path

random.seed(42)

# Unzip the downloaded dataset first
zip_path = Path("/content/plantdisease.zip")
extract_path = Path("/content/plant_data")

if zip_path.exists() and not extract_path.exists():
    print("Extracting dataset...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)
    print("Extraction complete!")

# Target directory pointing directly to the main PlantVillage folder inside plant_data
root = Path("/content/plant_data/PlantVillage")

selected = [
    "Tomato_Bacterial_spot",
    "Tomato_Early_blight",
    "Tomato_Late_blight",
    "Tomato_healthy"
]

# Find class folders directly in root
class_dirs = {}
for name in selected:
    folder = root / name
    if folder.is_dir():
        class_dirs[name] = folder

print("Found classes:", list(class_dirs.keys()))

assert len(class_dirs) == 4, (
    f"All four classes were not found. Expected {selected} inside {root}"
)

out = Path("/content/plant_split")
shutil.rmtree(out, ignore_errors=True)

for split in ["train", "validation", "test"]:
    for cls in selected:
        (out / split / cls).mkdir(parents=True, exist_ok=True)

for cls, folder in class_dirs.items():
    images = [
        p for p in folder.iterdir()
        if p.is_file() and p.suffix.lower()
        in [".jpg", ".jpeg", ".png"]
    ]

    random.shuffle(images)
    n = len(images)
    train_end = int(0.70 * n)
    val_end = int(0.85 * n)

    groups = {
        "train": images[:train_end],
        "validation": images[train_end:val_end],
        "test": images[val_end:]
    }

    for split, paths in groups.items():
        for i, path in enumerate(paths):
            dest = out / split / cls / f"{i}_{path.name}"
            shutil.copy2(path, dest)

print("Train, validation and test datasets ready!")

Found classes: ['Tomato_Bacterial_spot', 'Tomato_Early_blight', 'Tomato_Late_blight', 'Tomato_healthy']
Train, validation and test datasets ready!


In [ ]:
from pathlib import Path

root = Path("/content/plant_data")
# Let's inspect the first 20 directories inside plant_data to find the correct structure
subfolders = [p for p in root.rglob("*") if p.is_dir()]
print(f"Total subdirectories found: {len(subfolders)}")
print("Sample of subdirectories:")
for f in subfolders[:30]:
    print(f.relative_to(root))

Total subdirectories found: 33
Sample of subdirectories:
PlantVillage
plantvillage
PlantVillage/Pepper__bell___healthy
PlantVillage/Tomato_Bacterial_spot
PlantVillage/Tomato_healthy
PlantVillage/Pepper__bell___Bacterial_spot
PlantVillage/Potato___Late_blight
PlantVillage/Tomato__Target_Spot
PlantVillage/Tomato_Late_blight
PlantVillage/Tomato_Spider_mites_Two_spotted_spider_mite
PlantVillage/Tomato_Septoria_leaf_spot
PlantVillage/Tomato_Early_blight
PlantVillage/Tomato_Leaf_Mold
PlantVillage/Potato___healthy
PlantVillage/Tomato__Tomato_mosaic_virus
PlantVillage/Tomato__Tomato_YellowLeaf__Curl_Virus
PlantVillage/Potato___Early_blight
plantvillage/PlantVillage
plantvillage/PlantVillage/Pepper__bell___healthy
plantvillage/PlantVillage/Tomato_Bacterial_spot
plantvillage/PlantVillage/Tomato_healthy
plantvillage/PlantVillage/Pepper__bell___Bacterial_spot
plantvillage/PlantVillage/Potato___Late_blight
plantvillage/PlantVillage/Tomato__Target_Spot
plantvillage/PlantVillage/Tomato_Late_blight
pl

In [ ]:
import tensorflow as tf
import json

IMG_SIZE = (128, 128)
BATCH_SIZE = 32
SEED = 42

train_ds = tf.keras.utils.image_dataset_from_directory(
    "/content/plant_split/train",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    seed=SEED
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    "/content/plant_split/validation",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    "/content/plant_split/test",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

class_names = train_ds.class_names
num_classes = len(class_names)

with open("/content/class_names.json", "w") as f:
    json.dump(class_names, f)

AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.cache().shuffle(1000).prefetch(AUTOTUNE)
val_ds = val_ds.cache().prefetch(AUTOTUNE)
test_ds = test_ds.cache().prefetch(AUTOTUNE)

data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.08),
    tf.keras.layers.RandomZoom(0.1)
])

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(128, 128, 3)),
    data_augmentation,
    tf.keras.layers.Rescaling(1./255),

    tf.keras.layers.Conv2D(
        32, 3, activation="relu", padding="same"
    ),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Conv2D(
        64, 3, activation="relu", padding="same"
    ),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Conv2D(
        128, 3, activation="relu", padding="same"
    ),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dropout(0.4),
    tf.keras.layers.Dense(num_classes, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=4,
        restore_best_weights=True
    )
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=callbacks
)

model.save("/content/plant_disease_model.keras")

print("Model trained and saved!")

Found 4637 files belonging to 4 classes.
Found 994 files belonging to 4 classes.
Found 996 files belonging to 4 classes.
Epoch 1/20
145/145 ━━━━━━━━━━━━━━━━━━━━ 14s 34ms/step - accuracy: 0.5607 - loss: 1.0366 - val_accuracy: 0.7958 - val_loss: 0.6086
Epoch 2/20
145/145 ━━━━━━━━━━━━━━━━━━━━ 11s 22ms/step - accuracy: 0.7770 - loss: 0.5854 - val_accuracy: 0.8883 - val_loss: 0.3297
Epoch 3/20
145/145 ━━━━━━━━━━━━━━━━━━━━ 3s 22ms/step - accuracy: 0.8124 - loss: 0.5032 - val_accuracy: 0.8310 - val_loss: 0.4304
Epoch 4/20
145/145 ━━━━━━━━━━━━━━━━━━━━ 3s 22ms/step - accuracy: 0.8449 - loss: 0.4127 - val_accuracy: 0.8843 - val_loss: 0.3065
Epoch 5/20
145/145 ━━━━━━━━━━━━━━━━━━━━ 3s 21ms/step - accuracy: 0.8607 - loss: 0.3582 - val_accuracy: 0.8642 - val_loss: 0.3849
Epoch 6/20
145/145 ━━━━━━━━━━━━━━━━━━━━ 3s 22ms/step - accuracy: 0.8831 - loss: 0.3108 - val_accuracy: 0.8974 - val_loss: 0.3349
Epoch 7/20
145/145 ━━━━━━━━━━━━━━━━━━━━ 3s 22ms/step - accuracy: 0.8769 - loss: 0.3077 - val_accuracy: 

In [ ]:
from sklearn.metrics import classification_report
import numpy as np

loss, accuracy = model.evaluate(test_ds)

print("Test Loss:", loss)
print("Test Accuracy:", round(accuracy * 100, 2), "%")

y_true = np.concatenate([
    labels.numpy() for _, labels in test_ds
])

y_prob = model.predict(test_ds)
y_pred = np.argmax(y_prob, axis=1)

print(classification_report(
    y_true,
    y_pred,
    target_names=class_names
))

32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 43ms/step - accuracy: 0.9709 - loss: 0.0773
Test Loss: 0.07730062305927277
Test Accuracy: 97.09 %
32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step
                       precision    recall  f1-score   support

Tomato_Bacterial_spot       0.99      0.97      0.98       320
  Tomato_Early_blight       0.90      0.95      0.93       150
   Tomato_Late_blight       0.97      0.96      0.96       287
       Tomato_healthy       0.99      1.00      1.00       239

             accuracy                           0.97       996
            macro avg       0.96      0.97      0.97       996
         weighted avg       0.97      0.97      0.97       996



In [ ]:
from google.colab import files

files.download("/content/plant_disease_model.keras")
files.download("/content/class_names.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [4]:
import pickle
import json
from google.colab import files

# 1. Save the trained model in Pickle format
pickle_bundle = {
    "model_json": model.to_json(),
    "weights": model.get_weights(),
    "class_names": class_names,
    "img_size": IMG_SIZE
}

with open("plant_disease_model.pkl", "wb") as f:
    pickle.dump(pickle_bundle, f)

# 2. Save model metadata separately
metadata = {
    "class_names": class_names,
    "img_size": IMG_SIZE
}

with open("plant_disease_metadata.pkl", "wb") as f:
    pickle.dump(metadata, f)

# 3. Download both files
files.download("plant_disease_model.pkl")
files.download("plant_disease_metadata.pkl")

print("Pickle files created successfully!")

NameError: name 'model' is not defined